In [1]:
import torch
import os
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import copy
from pathlib import Path
import torch.profiler
from enum import Enum
import optuna
from sklearn.model_selection import GroupShuffleSplit
import random
from sklearn.metrics import (
    accuracy_score,
    f1_score,
)
import polars as pl

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [2]:
def seed_all(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all()

In [3]:
class ModelType(Enum):
    CNN = 1
    LSTM = 2
    CNN_LSTM = 3
    LSTM_CNN = 4
    CNN_LSTM_Fusion = 5
    CNN_Transformer = 6
    MLP = 7

class EvalMetric(Enum):
    accuracy = 1
    F1 = 2
    custom = 3

class ExperimentType(Enum):
    fixed_model_params = 1
    best_model = 2

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.backends.cudnn.benchmark = True

print(device)

cuda


In [5]:
root = "../../../"
data_path = f'{root}data/'
training_data_path = data_path + "Final Training Data/"

In [6]:
class CNN(nn.Module):
    def __init__(self, input_channels, output_channels, num_classes, activation_fn, dropout=0.2):
        super().__init__()

        C = output_channels

        self.features = nn.Sequential(
            nn.Conv1d(input_channels, C, 5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),

            nn.Conv1d(C, C, 5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),

            nn.Conv1d(C, C, 3, padding=1),
            nn.BatchNorm1d(C),
            activation_fn(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(C, C),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(C, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)



        
class LSTM(nn.Module):
    def __init__(self, input_size, num_classes, dropout=0.2):
        super().__init__()

        H = 130  

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=H,
            num_layers=1,   
            batch_first=True,
            dropout=0
        )

        self.head = nn.Sequential(
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        return self.head(x)

class AttentionPooling(nn.Module):
    """
    Learns which time steps are important instead of averaging blindly.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        weights = self.attn(x)             
        weights = torch.softmax(weights, dim=1)

        pooled = torch.sum(x * weights, dim=1)
        return pooled





class CNN_LSTM(nn.Module):
    def __init__(self, input_channels, num_classes, activation_fn, dropout=0.3):
        super().__init__()

        C = 96   # stronger feature space than 80
        H = 128  # stronger temporal embedding

        # ---------------- CNN FEATURE EXTRACTOR ----------------
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C, kernel_size=5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),
            nn.Dropout(0.1),

            nn.Conv1d(C, C, kernel_size=5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),
            nn.Dropout(0.1),

            nn.Conv1d(C, C, kernel_size=3, padding=1),
            nn.BatchNorm1d(C),
            activation_fn(),

            nn.MaxPool1d(2)
        )

        # ---------------- LSTM ----------------
        self.lstm = nn.LSTM(
            input_size=C,
            hidden_size=H,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.norm = nn.LayerNorm(H)

        # ---------------- POOLING ----------------
        self.pool = AttentionPooling(H)

        # ---------------- CLASSIFIER HEAD ----------------
        self.head = nn.Sequential(
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(H, H // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(H // 2, num_classes)
        )

    def forward(self, x):
        # x: (B, T, F)

        x = x.permute(0, 2, 1)  

        x = self.cnn(x)         

        x = x.permute(0, 2, 1)  

        x, _ = self.lstm(x)      

        x = self.norm(x)

        x = self.pool(x)         

        return self.head(x)




class CNN_Transformer(nn.Module):
    def __init__(self, input_channels, output_channels, num_layers, num_classes, feature_dim, activation_fn, dropout=0.3):
        super().__init__()

        C = output_channels

        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C, 5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),

            nn.Conv1d(C, C, 5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),

            nn.MaxPool1d(2)
        )

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=C,
                nhead=4,
                batch_first=True
            ),
            num_layers=num_layers
        )

        self.fusion = nn.Sequential(
            nn.Linear(C + feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, x_engineered):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)

        x = self.transformer(x)
        x = x.mean(dim=1)

        x = torch.cat([x, x_engineered], dim=1)
        return self.fusion(x)



        
class LSTM_CNN(nn.Module):
    def __init__(self, input_size, num_classes, activation_fn, dropout=0.2):
        super().__init__()

        H = 85  

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=H,
            num_layers=1,
            batch_first=True
        )

        self.cnn = nn.Sequential(
            nn.Conv1d(H, H, 5, padding=2),
            nn.BatchNorm1d(H),
            activation_fn(),

            nn.MaxPool1d(2),

            nn.Conv1d(H, H, 3, padding=1),
            activation_fn(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(H, H),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)





class CNN_LSTM_Fusion(nn.Module):
    def __init__(self, input_channels, output_channels, hidden_size, num_layers, num_classes, activation_fn, dropout=0.2):
        super().__init__()

        C = output_channels
        H = hidden_size  

        # CNN branch
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C, 5, padding=2),
            nn.BatchNorm1d(C),
            activation_fn(),

            nn.MaxPool1d(2),
        )

        # LSTM branch
        self.lstm = nn.LSTM(
            input_size=input_channels,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True
        )

        # fusion head
        self.classifier = nn.Sequential(
            nn.Linear(C + H, 96),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(96, num_classes)
        )

    def forward(self, x):
        # CNN
        x_cnn = x.permute(0, 2, 1)
        x_cnn = self.cnn(x_cnn)
        x_cnn = x_cnn.mean(dim=-1)

        # LSTM
        x_lstm, _ = self.lstm(x)
        x_lstm = x_lstm.mean(dim=1)

        x = torch.cat([x_cnn, x_lstm], dim=1)
        return self.classifier(x)



        
class MLP(nn.Module):
    def __init__(self, input_features, num_classes, activation_fn, dropout=0.3):
        super().__init__()

        H = 128

        self.net = nn.Sequential(
            nn.Linear(input_features, H),
            nn.LayerNorm(H),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Linear(H, H),
            nn.LayerNorm(H),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [8]:
def compute_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def compute_f1_score(y_true, y_pred):
    return f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
    )

In [9]:
def load_npz(path):
    path = Path(path)
    data = np.load(f"{path}.npz", allow_pickle=True)
    return data["X"], data["y"]

def load_subject(cache_dir, subject, use_raw, use_engineered):
    X_raw = None
    X_engineered = None
    y = None

    cache_dir = Path(cache_dir)
    if use_raw:
        X_raw, y = load_npz(cache_dir / subject / f"{subject}_raw")

    if use_engineered:
        X_engineered, y_engineered = load_npz(cache_dir / subject / f"{subject}_extracted")

        if y is None:
            y = y_engineered
        elif not np.array_equal(y, y_engineered):
            raise ValueError("Labels do not match")

    return X_raw, X_engineered, y

def get_subjects(cache_dir, use_raw, use_engineered):
    cache_dir = Path(cache_dir)
    subjects = set()

    if use_raw:
        subjects.update(
            p.stem.removesuffix("_raw")
            for p in Path(cache_dir).rglob("*_raw.npz")
        )

    if use_engineered:
        subjects.update(
            p.stem.removesuffix("_extracted")
            for p in Path(cache_dir).rglob("*_extracted.npz")
        )

    return sorted(subjects)

In [10]:
def get_loader(settings, X_train_raw, X_train_extracted, y_train):
    sampler = None
    shuffle = True
    batch_size = settings["batch size"]
    if settings["weight"] == "weighted":
            
        class_counts = np.bincount(y_train)
        weights = class_counts.sum() / class_counts
        
        weights = weights / weights.mean()
        #weights = 1.0 / class_counts
        
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    else:
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing = settings["label smoothing"])

    if X_train_raw is None:
        X_train_raw = torch.empty((len(y_train), 0, 0), dtype=torch.float32)

    if X_train_extracted is None:
        X_train_extracted = torch.empty((len(y_train), 0), dtype=torch.float32)

    return DataLoader(
                    TensorDataset(X_train_raw, X_train_extracted, y_train),
                    batch_size=batch_size,
                    shuffle=shuffle,
                    sampler=sampler,
                    #num_workers=2,
                    pin_memory=True,
                    #prefetch_factor=2
                ), loss_fn

In [11]:
# Gradient Accumulation
def train_epoch(model, loader, optimizer, loss_fn, accumulation_steps=64):
    model.train()
    optimizer.zero_grad()
    for i, (X_batch_raw, X_batch_extracted, y_batch) in enumerate(loader):
        X_batch_raw = X_batch_raw.to(device, non_blocking=True)
        X_batch_extracted = X_batch_extracted.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        torch.compiler.cudagraph_mark_step_begin()
        
        with torch.amp.autocast(device_type="cuda"):
            logits = model_forward(model, X_batch_raw, X_batch_extracted)
            loss = loss_fn(logits, y_batch)
            loss = loss / accumulation_steps
        
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

def validate_epoch(model, X_val_raw, X_val_extracted, y_val, metadata):

    model.eval()

    with torch.no_grad():

        X_val_raw = X_val_raw.to(device)
        X_val_extracted = X_val_extracted.to(device)
        y_val = y_val.to(device)

        logits = model_forward(
            model,
            X_val_raw,
            X_val_extracted
        )

        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(probs, dim=1)
        preds = preds.cpu().numpy()

        if metadata["eval_metric"] == EvalMetric.accuracy:
            val_metric = compute_accuracy(
                y_val.cpu(),
                preds
            )

        elif metadata["eval_metric"] == EvalMetric.F1:
            val_metric = compute_f1_score(
                y_val.cpu(),
                preds
            )
        else:
            confidences, preds = torch.max(probs, dim=1)

            mask = confidences >= 0.8

            if mask.sum() == 0:
                val_metric = 0
            else:
                coverage = mask.float().mean().item()
                precision = (
                    preds[mask] == y_val[mask]
                ).float().mean().item()

                val_metric = precision * coverage

    return val_metric

def model_forward(model, x_raw, x_extracted):

    has_raw = x_raw.numel() > 0
    has_extracted = x_extracted.numel() > 0

    if has_raw and has_extracted:
        return model(x_raw, x_extracted)

    if has_raw:
        return model(x_raw)

    if has_extracted:
        return model(x_extracted)

    raise ValueError("No input features provided")

In [12]:
def get_model(model_settings, X_train_main_raw, X_train_main_extracted, y_train_main):
    match model_settings["activation_fn"]:
        case "relu":
            activation_fn = nn.ReLU
        case "gelu":
            activation_fn = nn.GELU

    dropout = model_settings["dropout"]
    hidden_size = model_settings["hidden_size"]
    num_layers = model_settings["num_layers"]
    output_channles = model_settings["output_channels"]

    input_channels_raw = X_train_main_raw.shape[-1]
    input_channels_extracted = X_train_main_extracted.shape[1]
    num_classes = len(torch.unique(y_train_main))
    
    match model_settings["model"]:
        case ModelType.CNN:
            model = CNN(
                input_channels_raw,
                output_channles,
                num_classes,
                activation_fn
            ).to(device)
        case ModelType.LSTM:
            model = LSTM(
                input_channels_raw,
                num_classes,
            ).to(device)
        case ModelType.CNN_LSTM:
            model = CNN_LSTM(
                input_channels_raw, 
                num_classes,
                activation_fn,
                dropout
            ).to(device)
        case ModelType.LSTM_CNN:
            model = LSTM_CNN(
                input_channels_raw,
                num_classes,
                activation_fn,
                dropout
            ).to(device)
        case ModelType.CNN_Transformer:
            model = CNN_Transformer(
                input_channels_raw,
                output_channles,
                num_layers,
                num_classes,
                input_channels_extracted,
                activation_fn,
                dropout
            ).to(device)
        case ModelType.CNN_LSTM_Fusion:
            model = CNN_LSTM_Fusion(
                input_channels_raw,
                output_channles,
                hidden_size,
                num_layers,
                num_classes,
                activation_fn,
                dropout
            ).to(device)
        case ModelType.MLP:
            model = MLP(
                input_channels_extracted,
                num_classes,
                activation_fn,
                dropout
            ).to(device)

    return model

In [13]:
def train_loso(
    X_train_raw, X_test_raw,
    X_train_extracted, X_test_extracted,
    y_train, y_test, 
    groups,
    subject_idx, test_subject,
    settings, metadata,
):
    # ------------------------------------------------------------------
    # Convert to CPU tensors
    # ------------------------------------------------------------------
    if X_train_raw is not None:
        X_train_raw = torch.tensor(X_train_raw, dtype=torch.float32)
        X_test_raw = torch.tensor(X_test_raw, dtype=torch.float32)

    if X_train_extracted is not None:
        X_train_extracted = torch.tensor(X_train_extracted, dtype=torch.float32)
        X_test_extracted = torch.tensor(X_test_extracted, dtype=torch.float32)

    y_train = torch.tensor(y_train, dtype=torch.long)

    # ------------------------------------------------------------------
    # Split train / validation
    # ------------------------------------------------------------------
    split_source = X_train_raw if X_train_raw is not None else X_train_extracted

    split_np = split_source.numpy()
    y_train_np = y_train.numpy()

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42 + subject_idx,
    )

    train_idx, val_idx = next(
        gss.split(split_np, y_train_np, groups)
    )

    assert len(groups) == len(split_np)

    # ------------------------------------------------------------------
    # Slice data
    # ------------------------------------------------------------------
    X_train_main_raw = None
    X_val_raw = None

    X_train_main_extracted = None
    X_val_extracted = None

    if X_train_raw is not None:
        X_train_main_raw = X_train_raw[train_idx]
        X_val_raw = X_train_raw[val_idx]

    if X_train_extracted is not None:
        X_train_main_extracted = X_train_extracted[train_idx]
        X_val_extracted = X_train_extracted[val_idx]

    y_train_main = y_train[train_idx]
    y_val = y_train[val_idx]

    if metadata["eval_metric"] == EvalMetric.custom:
        y_val = y_val.float()


    if X_train_main_raw is None:
        X_train_main_raw = torch.empty(
            (len(y_train_main), 0, 0),
            dtype=torch.float32
        )
    
    if X_val_raw is None:
        X_val_raw = torch.empty(
            (len(y_val), 0, 0),
            dtype=torch.float32
        )
    
    if X_train_main_extracted is None:
        X_train_main_extracted = torch.empty(
            (len(y_train_main), 0),
            dtype=torch.float32
        )
    
    if X_val_extracted is None:
        X_val_extracted = torch.empty(
            (len(y_val), 0),
            dtype=torch.float32
        )
    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    model = get_model(
        settings["model settings"],
        X_train_main_raw,
        X_train_main_extracted,
        y_train_main,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=settings["training settings"]["learning rate"],
        weight_decay=settings["training settings"]["weight decay"],
    )

    loader, loss_fn = get_loader(
        settings["training settings"],
        X_train_main_raw, X_train_main_extracted, y_train_main,
    )

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    best_val = -1
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(settings["training settings"]["epoch"]):

        train_epoch(
            model,loader, optimizer, loss_fn,
            settings["training settings"]["accumulation_steps"],
        )

        val_score = validate_epoch(
            model,
            X_val_raw, X_val_extracted, y_val,
            metadata
        )

        if val_score > best_val:
            best_val = val_score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= settings["training settings"]["patience"]:
            break

    print(epoch - patience_counter)

    model.load_state_dict(best_state)

    # ------------------------------------------------------------------
    # Test
    # ------------------------------------------------------------------
    model.eval()

    with torch.no_grad():

        if X_test_raw is None:
            X_test_raw = torch.empty((len(y_test), 0, 0))
        
        if X_test_extracted is None:
            X_test_extracted = torch.empty((len(y_test), 0))
            
        y_test = torch.tensor(y_test, dtype=torch.float32)
        
        logits = model_forward(
            model,
            X_test_raw.to(device),
            X_test_extracted.to(device),
        )

        prob = torch.softmax(logits, dim=1).cpu()
        
        _, preds = torch.max(prob, dim=1)

        preds = preds.cpu()
        y_test = y_test.cpu()
        
        acc = compute_accuracy(y_test,preds)

    return acc

In [14]:
def normalize_train_test_raw(X_train, X_test, eps=1e-8):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)

    X_train = (X_train - mean) / (std + eps)
    X_test = (X_test - mean) / (std + eps)

    return X_train, X_test

def normalize_train_test_engineered(X_train, X_test, eps=1e-8):
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)

    X_train = (X_train - mean) / (std + eps)
    X_test = (X_test - mean) / (std + eps)

    return X_train, X_test

In [15]:
def save_results(study, trial):
    study.trials_dataframe().to_csv(
        f"{study_name}_trials.csv",
        index=False
    )

In [16]:
def printParmNum(X_train, X_test, y_train, y_test, groups, subject_idx, settings):
    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, dtype=torch.long, device=device)

    X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_test = torch.tensor(y_test, dtype=torch.long, device=device)

    # ----------------------------
    # validation split
    # ----------------------------
    X_train_np = X_train.cpu().numpy()
    y_train_np = y_train.cpu().numpy()

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42 + subject_idx
    )

    train_idx, val_idx = next(
        gss.split(X_train_np, y_train_np, groups)
    )
    
    assert len(groups) == len(X_train_np)
    
    X_train_main = X_train_np[train_idx]
    y_train_main = y_train_np[train_idx]

    X_val = X_train_np[val_idx]
    y_val = y_train_np[val_idx]

    X_train_main = torch.tensor(X_train_main, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)

    y_train_main = torch.tensor(y_train_main, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)

    model_settings = settings["model settings"]
    training_settings = settings["training settings"]
    
    lr_range = training_settings["learning rate"]
    wd_range = training_settings["weight decay"]
    
    match model_settings["activation_fn"]:
        case "relu":
            activation_fn = nn.ReLU
        case "gelu":
            activation_fn = nn.GELU

    model = CNN(
        X_train.shape[-1],
        num_classes=len(torch.unique(y_train)),
        activation_fn=activation_fn
    ).to(device)
    print("CNN: ")
    print(f"\t{count_parameters(model)} Params")

    model = LSTM(
        X_train.shape[-1],
        num_classes=len(torch.unique(y_train)),
    ).to(device)
    print("LSTM: ")
    print(f"\t{count_parameters(model)} Params")
    model = CNN_LSTM(
        X_train.shape[-1], 
        len(torch.unique(y_train)),
        activation_fn=activation_fn,
        dropout=model_settings["dropout"]
    ).to(device)
    print("CNN_LSTM: ")
    print(f"\t{count_parameters(model)} Params")

    model = LSTM_CNN(
        X_train.shape[-1],
        len(torch.unique(y_train)),
        activation_fn=activation_fn,
        dropout=model_settings["dropout"]
    ).to(device)
    print("LSTM_CNN: ")
    print(f"\t{count_parameters(model)} Params")

    model = CNN_LSTM_Fusion(
        X_train.shape[-1],
        len(torch.unique(y_train)),
        activation_fn=activation_fn,
        dropout=model_settings["dropout"]
    ).to(device)
    print("CNN_LSTM_Fusion: ")
    print(f"\t{count_parameters(model)} Params")

In [17]:
study_name = "cnn_full_data_search"

In [ ]:
def objective(trial):
    metadata = {
        "eval_metric": EvalMetric.accuracy,
    }


    
    # settings = {
    #     "training settings": {
    #         "learning rate": trial.suggest_float("lr", 0.00001, 0.0001, log=True),
    #         "weight decay": trial.suggest_float("wd", 0.0005, 0.0022, log=True),
    
    #         "patience": 15,
    #         "epoch": 150,
            
    #         "weight": None,
    #         "label smoothing": trial.suggest_float("ls", 0, 1),
    #         "batch size": trial.suggest_categorical("batch", [64, 128, 256, 512]),
    #         "accumulation_steps": 10,
    #     },
    #     "model settings": {
    #         "model": ModelType.CNN_LSTM_Fusion,
    #         "activation_fn": "gelu",
    #         "dropout": trial.suggest_float("dp", 0.1, 0.35), 
    #         "output_channels": trial.suggest_categorical("out_ch", [32, 64, 128, 256, 512]),
    #         "hidden_size": trial.suggest_categorical("h_size", [32, 64, 128, 256, 512, 1024]),
    #         "num_layers": 2,
    #     }
    # }

    settings = {
        "training settings": {
            "learning rate": trial.suggest_float("lr", 0.0005, 0.0025, log=True),
            "weight decay": trial.suggest_float("wd", 0.001, 0.0032, log=True),
    
            "patience": 15,
            "epoch": 150,
            
            "weight": None,
            "label smoothing": trial.suggest_float("ls", 0, 0.72),
            "batch size": trial.suggest_categorical("batch", [64, 128, 256, 512]),
            "accumulation_steps": 10,
        },
        "model settings": {
            "model": ModelType.CNN,
            "activation_fn": "gelu",
            "dropout": trial.suggest_float("dp", 0.3, 0.8), 
            "output_channels": trial.suggest_categorical("out_ch", [32, 64, 128, 256, 512]),
            "hidden_size": 0,
            "num_layers": 0,
        }
    }
    
    # settings = {
    #     "training settings": {
    #         "learning rate": trial.suggest_float("lr", 0.00005, 0.006, log=True),
    #         "weight decay": trial.suggest_float("wd", 0.00205, 0.0050, log=True),
    
    #         "patience": 15,
    #         "epoch": 150,
            
    #         "weight": None,
    #         "label smoothing": trial.suggest_float("ls", 0.2882, 0.8022),
    #         "batch size": trial.suggest_categorical("batch", [64, 128, 256, 512]),
    #         "accumulation_steps": 10,
    #     },
    #     "model settings": {
    #         "model": ModelType.CNN_Transformer,
    #         "activation_fn": "gelu",
    #         "dropout": trial.suggest_float("dp", 0.62, 1), 
    #         "output_channels": trial.suggest_categorical("out_ch", [32, 64, 128, 256, 512]),
    #         "hidden_size": trial.suggest_categorical("h_size", [32, 64, 128, 256, 512, 1024]),
    #         "num_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
    #     }
    # }

    model_type = settings["model settings"]["model"]
    use_raw = model_type in [
        ModelType.CNN,
        ModelType.LSTM,
        ModelType.CNN_LSTM,
        ModelType.LSTM_CNN,
        ModelType.CNN_Transformer,
        ModelType.CNN_LSTM_Fusion,
    ]
    
    use_engineered = model_type in [
        ModelType.MLP,
        ModelType.CNN_Transformer,
    ]
    files_suffix = "_raw" if use_raw else "_extracted"
    
    cache_hash = "7f614e721ed6b83d1d2a0c91f5a7c0ef"
    cache_dir = f"{data_path}Final Training Data/Windowed Data/{cache_hash}"
    
    subjects = get_subjects(cache_dir, use_raw, use_engineered)

    accs = []
    for subject_idx, test_subject in enumerate(subjects):
    
        train_subjects = [s for s in subjects if s != test_subject]
        test_subject = test_subject.removesuffix(files_suffix)
        print(f"\nSubject {subject_idx}: {test_subject}")
        print("Loading data...")
    
        X_train_raw_list = []
        X_train_engineered_list = []
        y_train_list = []
        groups = []
        
        for s in train_subjects:
            s = s.removesuffix(files_suffix)
            X_s_raw, X_s_engineered, y_s = load_subject(cache_dir, s, use_raw, use_engineered)
            if use_raw:
                X_train_raw_list.append(X_s_raw)
            
            if use_engineered:
                X_train_engineered_list.append(X_s_engineered)
            
            y_train_list.append(y_s)
            groups.extend([s] * len(X_s_raw if X_s_raw is not None else X_s_engineered))
            
        X_train_raw = np.concatenate(X_train_raw_list) if use_raw else None
        X_train_engineered = np.concatenate(X_train_engineered_list) if use_engineered else None
        y_train = np.concatenate(y_train_list)
        groups = np.array(groups)
        
        X_test_raw, X_test_engineered, y_test = load_subject(cache_dir, test_subject, use_raw, use_engineered)
        if use_raw:
            X_train_raw, X_test_raw = normalize_train_test_raw(
                X_train_raw,
                X_test_raw
            )
        
        if use_engineered:
            X_train_engineered, X_test_engineered = normalize_train_test_engineered(
                X_train_engineered,
                X_test_engineered
            )
        
        print("Training...")
        acc = train_loso(
                X_train_raw, X_test_raw,
                X_train_engineered, X_test_engineered,
                y_train, y_test,
                groups,
                subject_idx, test_subject,
                settings, metadata
            )
        accs.append(acc)
    return sum(accs) / len(accs)



study = optuna.create_study(
    study_name=study_name,
    storage="sqlite:///optuna.db",
    load_if_exists=True,
    direction="maximize",
)
study.optimize(objective, n_trials=100, callbacks=[save_results])

[I 2026-07-22 07:59:31,260] Using an existing study with name 'cnn_full_data_search' instead of creating a new one.



Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
2

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
5

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
1

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
3

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
0

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
6

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
2

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
4

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
0

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
2

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
1

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
0

Subject 12: 762be8f2-4fec-4ea5-bdbe-6aee8697b9ea


[I 2026-07-22 08:05:03,738] Trial 101 finished with value: 0.4991221766400664 and parameters: {'lr': 0.002414672687717356, 'wd': 0.0023058775046019965, 'ls': 0.6516669389148477, 'batch': 512, 'dp': 0.5777171402220671, 'out_ch': 32}. Best is trial 44 with value: 0.5231191890031475.



Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
1

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
4

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
2

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
1

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
38

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
7

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
2

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
3

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
0

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
1

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
2

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
0

Subject 12: 762be8f2-4fec-4ea5-bdbe-6aee8697b9ea

[I 2026-07-22 08:10:50,392] Trial 102 finished with value: 0.49955203303061907 and parameters: {'lr': 0.0007262965705843596, 'wd': 0.001709725473942344, 'ls': 0.6751175968540025, 'batch': 512, 'dp': 0.5522359931972237, 'out_ch': 64}. Best is trial 44 with value: 0.5231191890031475.



Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
10

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
2

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
2

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
1

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
7

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
1

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
4

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
15

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
17

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
4

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
20

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
9

Subject 12: 762be8f2-4fec-4ea5-bdbe-6aee8697b

[I 2026-07-22 08:16:43,836] Trial 103 finished with value: 0.5033122620411865 and parameters: {'lr': 0.000594044177117317, 'wd': 0.0015103543203063155, 'ls': 0.6951410750226492, 'batch': 512, 'dp': 0.6062220619896266, 'out_ch': 32}. Best is trial 44 with value: 0.5231191890031475.


14

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
1

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
0

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
0

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
15

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
17

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
10

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
17

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
1

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
17

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
0

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
5

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
0

Subject 12: 762be8f2-4fec-4ea5-bdbe-6aee8

[I 2026-07-22 08:22:26,255] Trial 104 finished with value: 0.49414051360059047 and parameters: {'lr': 0.0020779037913167073, 'wd': 0.001595723992386061, 'ls': 0.5226994928368394, 'batch': 512, 'dp': 0.5470805117019257, 'out_ch': 64}. Best is trial 44 with value: 0.5231191890031475.


4

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...


In [ ]:
CSV_PATH = f"{study_name}_trials.csv"
SCORE_COLUMN = "value"
 
df = pl.read_csv(CSV_PATH)

# Keep only completed trials (if the column exists)
if "state" in df.columns:
    df = df.filter(pl.col("state") == "COMPLETE")

best_values = df[ df["value"].arg_max() ]
best_score = best_values.select(pl.col("value")).item()
tolerance = 0.000

while True:
    best_df = df.filter(pl.col("value") >= best_score - tolerance)

    if best_df.height >= 10:
        break

    tolerance += 0.001

best_df = df.filter(
    pl.col(SCORE_COLUMN) >= best_score - tolerance
)

param_columns = [c for c in best_values.columns if c.startswith("params_")]

print(f"Best score: {best_score:.6f}")
for param in param_columns:
    val = (
        best_values.select(
            pl.col(param),
        )
        .row(0)
    )

    print(f"{param.removeprefix('params_'):20} "
          f"{val}")

print(f"Trials kept: {best_df.height} / {df.height}")
print(f"Min score: {best_score - tolerance}")
param_columns = [c for c in df.columns if c.startswith("params_")]

print("\nParameter ranges:")
for param in param_columns:
    min_val, max_val = (
        best_df.select(
            pl.col(param).min().alias("min"),
            pl.col(param).max().alias("max"),
        )
        .row(0)
    )

    print(f"{param.removeprefix('params_'):20} "
          f"min={min_val}    max={max_val}")